# TRPA1 Antagonist Screen

Virtual screening pipeline for human TRPA1 (UniProt O75762) antagonists.

**Strategy:** Dock a compound library against PDB 6V9W (cryo-EM structure with co-crystallized antagonist A-967079), rescore top hits with GNINA, then validate dual-consensus hits by AF3 co-folding with the TM domain.

| Phase | Method | Scale | Output |
|-------|--------|-------|--------|
| 0 | Receptor prep + grid box | 1× | `chain_A.pdbqt`, box coords |
| 1a | AutoDock Vina screen | ~5–10K compounds | `docking_hits.csv` |
| 1b | GNINA rescore | top 500 | `gnina_scores.csv` |
| 1c | Triage | top 500 | `top_hits.csv` (~20 hits) |
| 2 | AF3 co-fold validation | top 20 | per-compound iptm scores |

**Run cells top to bottom.** Each section ends with a PASS/FAIL check.

---
## Section 0 — Setup

In [ ]:
import sys, os
from pathlib import Path

# Make sure we're running from the repo root
REPO = Path(os.getcwd())
if REPO.name == 'notebooks':
    REPO = REPO.parent
    os.chdir(REPO)
sys.path.insert(0, str(REPO))

import pandas as pd
import numpy as np
import py3Dmol
from Bio.PDB import PDBParser

RECEPTOR_DIR = REPO / 'results' / 'trpa1' / 'receptor'
DOCKING_DIR  = REPO / 'results' / 'trpa1' / 'docking'
AF3_DIR      = REPO / 'results' / 'trpa1' / 'af3_validation'
LIGAND_LIB   = REPO / 'ligands' / 'library'

for d in [RECEPTOR_DIR, DOCKING_DIR, AF3_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f'Repo root : {REPO}')
print(f'Receptor  : {RECEPTOR_DIR}')
print(f'Docking   : {DOCKING_DIR}')
print(f'Library   : {LIGAND_LIB} — exists: {LIGAND_LIB.exists()}')

In [ ]:
# Verify key imports for the docking pipeline
import importlib, subprocess

checks = {
    'pandas': 'pandas',
    'biopython': 'Bio',
    'rdkit': 'rdkit',
    'py3Dmol': 'py3Dmol',
}

all_ok = True
for name, mod in checks.items():
    try:
        importlib.import_module(mod)
        print(f'  ✓ {name}')
    except ImportError:
        print(f'  ✗ {name} — run: conda activate af3-pipeline')
        all_ok = False

# Check AutoDock Vina CLI
r = subprocess.run(['vina', '--version'], capture_output=True, text=True)
if r.returncode == 0:
    print(f'  ✓ vina ({r.stdout.strip()})')
else:
    print('  ✗ vina not found — install via conda: conda install -c conda-forge autodock-vina')
    all_ok = False

print('\n✓ PASS — environment ready' if all_ok else '\n✗ FAIL — fix missing packages before continuing')

---
## Section 1 — Download & inspect PDB 6V9W

**PDB 6V9W:** Human TRPA1, 3.0 Å cryo-EM, co-crystallized with A-967079 (selective antagonist).  
Chain A is one TRPA1 subunit. The A-967079 ligand defines the TM4–S5 / intracellular S6 pocket.

In [ ]:
import urllib.request

pdb_path = RECEPTOR_DIR / '6V9W.pdb'

if not pdb_path.exists():
    print('Downloading 6V9W from RCSB...')
    urllib.request.urlretrieve(
        'https://files.rcsb.org/download/6V9W.pdb',
        pdb_path
    )
    print(f'  Saved to {pdb_path}')
else:
    print(f'  Already present: {pdb_path}')

# Report chains and ligands
parser = PDBParser(QUIET=True)
structure = parser.get_structure('6V9W', pdb_path)
model = structure[0]

chains = list(model.get_chains())
print(f'\nChains in 6V9W: {[c.id for c in chains]}')

hetatms = set()
for chain in chains:
    for res in chain:
        if res.id[0] != ' ':   # HETATM residues have non-blank record type
            hetatms.add((chain.id, res.resname, res.id[1]))

print('\nHETATM residues (ligands/ions/water):')
for chain_id, resname, seqid in sorted(hetatms):
    if resname != 'HOH':
        print(f'  Chain {chain_id}  {resname:6s}  #{seqid}')

print('\n✓ PASS — structure downloaded and parsed')

In [ ]:
# 3D preview of chain A with the co-crystallized ligand highlighted
pdb_text = pdb_path.read_text()

view = py3Dmol.view(width=800, height=500)
view.addModel(pdb_text, 'pdb')

# Show only chain A as cartoon
view.setStyle({'chain': 'A'}, {'cartoon': {'color': 'steelblue', 'opacity': 0.8}})

# Ligand as sticks — update 'PLX' to the actual residue name from the cell above
LIGAND_RESNAME = 'PLX'   # <-- check output above and update if different
view.setStyle({'resn': LIGAND_RESNAME}, {'stick': {'colorscheme': 'orangeCarbon', 'radius': 0.3}})
view.addSurface(py3Dmol.SAS, {'opacity': 0.15, 'color': 'lightblue'}, {'chain': 'A'})

view.zoomTo({'resn': LIGAND_RESNAME})
view.show()
print(f'Blue = TRPA1 chain A  |  Orange sticks = {LIGAND_RESNAME} (A-967079 antagonist pocket)')

---
## Section 2 — Prepare receptor PDBQT

Converts chain A to the PDBQT format AutoDock Vina requires: adds hydrogens, computes partial charges, removes water.

In [ ]:
from src.docking.prepare import prepare_receptor

pdbqt_path = RECEPTOR_DIR / 'chain_A.pdbqt'

if not pdbqt_path.exists():
    print('Preparing chain A PDBQT...')
    prepare_receptor(
        str(pdb_path),
        chain_id='A',
        output_dir=str(RECEPTOR_DIR)
    )
else:
    print(f'Already exists: {pdbqt_path}')

size_mb = pdbqt_path.stat().st_size / 1e6
print(f'\n{pdbqt_path.name}  ({size_mb:.1f} MB)')

# Quick check: count ATOM lines
atom_lines = [l for l in pdbqt_path.read_text().splitlines() if l.startswith('ATOM')]
print(f'ATOM records: {len(atom_lines)}')

if len(atom_lines) > 100:
    print('✓ PASS — receptor prepared')
else:
    print('✗ FAIL — too few atoms; check prepare_receptor output above')

---
## Section 3 — Define docking grid box

The grid box tells Vina where to search. We center it on the A-967079 ligand centroid and add 8 Å padding on each side to give the docked molecule room to move.

In [ ]:
# Update LIGAND_RESNAME below to match what you saw in Section 1
LIGAND_RESNAME = 'PLX'   # <-- update if needed

lig_atoms = [
    atom
    for atom in structure.get_atoms()
    if atom.parent.resname == LIGAND_RESNAME
]

if not lig_atoms:
    print(f'✗ No atoms found for residue name "{LIGAND_RESNAME}".')
    print('  Update LIGAND_RESNAME to one of the HETATM residues printed in Section 1.')
else:
    coords = np.array([a.coord for a in lig_atoms])
    center = coords.mean(axis=0)
    extent = coords.max(axis=0) - coords.min(axis=0)
    box    = extent + 8.0   # 4 Å padding each side

    print(f'Ligand: {LIGAND_RESNAME}  ({len(lig_atoms)} atoms)')
    print(f'Grid center : x={center[0]:.2f}  y={center[1]:.2f}  z={center[2]:.2f}')
    print(f'Grid size   : x={box[0]:.2f}   y={box[1]:.2f}   z={box[2]:.2f}')

    # Save for use in later cells
    BOX = dict(
        center_x=float(center[0]), center_y=float(center[1]), center_z=float(center[2]),
        size_x=float(box[0]),   size_y=float(box[1]),   size_z=float(box[2])
    )
    import json
    box_file = RECEPTOR_DIR / 'docking_box.json'
    box_file.write_text(json.dumps(BOX, indent=2))
    print(f'\nSaved to {box_file}')
    print('\n✓ PASS — grid box defined')

In [ ]:
# Visualize: chain A + ligand + docking box wireframe
import json
BOX = json.loads((RECEPTOR_DIR / 'docking_box.json').read_text())

view = py3Dmol.view(width=800, height=500)
view.addModel(pdb_path.read_text(), 'pdb')
view.setStyle({'chain': 'A'}, {'cartoon': {'color': 'steelblue', 'opacity': 0.6}})
view.setStyle({'resn': LIGAND_RESNAME}, {'stick': {'colorscheme': 'orangeCarbon', 'radius': 0.35}})

# Draw docking box
view.addBox({
    'center': {'x': BOX['center_x'], 'y': BOX['center_y'], 'z': BOX['center_z']},
    'dimensions': {'w': BOX['size_x'], 'h': BOX['size_y'], 'd': BOX['size_z']},
    'color': 'yellow',
    'opacity': 0.2,
    'wireframe': True
})

view.zoomTo({'resn': LIGAND_RESNAME})
view.show()
print('Yellow box = docking search volume  |  Orange = A-967079 (pocket anchor)')

---
## Section 4 — Benchmark: HC-030031 (known antagonist)

Before screening unknowns, dock HC-030031 to validate the grid box.  
**Expected result:** vina_zscore ≤ −2.5. If it fails, the box is misplaced — adjust LIGAND_RESNAME or box padding and rerun Section 3.

HC-030031 SMILES (PubChem CID 2723949 — verify before use):  
`CC(C)c1ccc(NC(=O)Cn2cnc3c(=O)n(C)c(=O)n(C)c32)cc1`

In [ ]:
from src.docking.screen import dock_single
import json

BOX = json.loads((RECEPTOR_DIR / 'docking_box.json').read_text())

HC030031_SMILES = 'CC(C)c1ccc(NC(=O)Cn2cnc3c(=O)n(C)c(=O)n(C)c32)cc1'

print('Docking HC-030031 (known TRPA1 antagonist)...')
result = dock_single(
    receptor_pdbqt=str(RECEPTOR_DIR / 'chain_A.pdbqt'),
    smiles=HC030031_SMILES,
    compound_name='HC-030031',
    output_dir=str(DOCKING_DIR / 'benchmark'),
    **BOX
)

print(f"\nVina best affinity : {result['vina_affinity']:.2f} kcal/mol")
print(f"Vina Z-score       : {result.get('vina_zscore', 'not yet calculated — run triage first')}")

if result['vina_affinity'] < -6.0:
    print('\n✓ PASS — HC-030031 docks with reasonable affinity; box placement looks correct')
else:
    print('\n⚠ WARNING — weak affinity; check box placement in Section 3 before proceeding')

In [ ]:
# Visualize HC-030031 pose vs A-967079 (do they overlap in the same pocket?)
pose_file = DOCKING_DIR / 'benchmark' / 'HC-030031_pose.pdbqt'

if pose_file.exists():
    view = py3Dmol.view(width=800, height=500)
    view.addModel(pdb_path.read_text(), 'pdb')
    view.setStyle({'chain': 'A'}, {'cartoon': {'color': 'steelblue', 'opacity': 0.5}})
    view.setStyle({'resn': LIGAND_RESNAME}, {'stick': {'colorscheme': 'orangeCarbon', 'radius': 0.3}})

    pose_text = pose_file.read_text()
    view.addModel(pose_text, 'pdbqt')
    view.setStyle({'model': 1}, {'stick': {'colorscheme': 'greenCarbon', 'radius': 0.3}})

    view.zoomTo({'resn': LIGAND_RESNAME})
    view.show()
    print('Orange = A-967079 (crystal)  |  Green = HC-030031 docked pose')
    print('They should overlap substantially if the box is correct.')
else:
    print('Pose file not found — check that dock_single ran successfully above.')

---
## Section 5 — Phase 1a: Vina library screen

Screens the full compound library against chain A.  
**Runtime:** ~1–4 hours depending on library size.  
Run this cell and let it finish before proceeding to Section 6.

**Library options (pick one):**
- `ligands/library/` — existing PPI library (fast first pass, less optimal chemistry for a TRP channel)
- Enamine ion-channel focused set (~5–10K) — better TRP chemistry, download from Enamine academic portal
- `ligands/custom.smi` — your own SMILES (included automatically)

In [ ]:
from src.docking.screen import run_screen
import json

BOX = json.loads((RECEPTOR_DIR / 'docking_box.json').read_text())

# Check library exists
if not LIGAND_LIB.exists() or not any(LIGAND_LIB.iterdir()):
    print('✗ FAIL — ligands/library/ is empty.')
    print('  Options:')
    print('  1. Download Enamine ion-channel set to ligands/library/')
    print('  2. Add SMILES to ligands/custom.smi and screen that file instead')
else:
    n_ligands = sum(1 for _ in LIGAND_LIB.glob('*.sdf')) + \
                sum(1 for _ in LIGAND_LIB.glob('*.smi'))
    print(f'Library: {n_ligands} files in {LIGAND_LIB}')
    print(f'Output:  {DOCKING_DIR}')
    print('Starting Vina screen (this will take a while)...\n')

    hits_csv = run_screen(
        receptor_pdbqt=str(RECEPTOR_DIR / 'chain_A.pdbqt'),
        ligand_dir=str(LIGAND_LIB),
        output_dir=str(DOCKING_DIR),
        **BOX
    )

    df = pd.read_csv(hits_csv)
    print(f'\nScreen complete: {len(df)} compounds docked')
    print(f'Best affinity: {df["vina_affinity"].min():.2f} kcal/mol')
    print(df.nsmallest(10, 'vina_affinity')[['name', 'vina_affinity']].to_string(index=False))
    print('\n✓ PASS — Vina screen complete')

In [ ]:
# Score distribution plot
import matplotlib.pyplot as plt

df = pd.read_csv(DOCKING_DIR / 'docking_hits.csv')

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df['vina_affinity'], bins=60, color='steelblue', edgecolor='white', alpha=0.8)
ax.axvline(df['vina_affinity'].mean() - 2.5 * df['vina_affinity'].std(),
           color='tomato', linestyle='--', label='Z = −2.5 threshold')
ax.set_xlabel('Vina affinity (kcal/mol)')
ax.set_ylabel('Compounds')
ax.set_title('Vina score distribution — TRPA1 screen')
ax.legend()
plt.tight_layout()
plt.show()

z_hits = df[df['vina_zscore'] <= -2.5]
print(f'Compounds with vina_zscore ≤ −2.5: {len(z_hits)}')

---
## Section 6 — Phase 1b: GNINA rescore (top 500 Vina hits)

In [ ]:
from src.docking.gnina import rescore_top_hits

gnina_csv = rescore_top_hits(
    receptor_pdbqt=str(RECEPTOR_DIR / 'chain_A.pdbqt'),
    poses_dir=str(DOCKING_DIR),
    scores_csv=str(DOCKING_DIR / 'docking_hits.csv'),
    output=str(DOCKING_DIR.parent / 'gnina_scores.csv'),
    top_n=500
)

gnina_df = pd.read_csv(gnina_csv)
print(f'GNINA rescored: {len(gnina_df)} compounds')
print(gnina_df.nsmallest(10, 'gnina_score')[['name', 'gnina_score', 'vina_affinity']].to_string(index=False))
print('\n✓ PASS — GNINA rescore complete')

---
## Section 7 — Phase 1c: Triage — hit selection

Applies Z-score cutoff and consensus filter to produce the final shortlist.

In [ ]:
from src.docking.triage import triage_hits

top_hits_csv = triage_hits(
    scores_csv=str(DOCKING_DIR / 'docking_hits.csv'),
    gnina_csv=str(DOCKING_DIR.parent / 'gnina_scores.csv'),
    output=str(DOCKING_DIR.parent / 'top_hits.csv'),
    vina_z_cutoff=-2.5
)

hits = pd.read_csv(top_hits_csv)
consensus = hits[hits['consensus_flag'] == True]

print(f'Total hits (vina_zscore ≤ −2.5)  : {len(hits)}')
print(f'Consensus hits (Vina AND GNINA)  : {len(consensus)}')
print()
print(consensus[['name', 'vina_affinity', 'vina_zscore', 'gnina_score']]
      .sort_values('vina_zscore')
      .head(20)
      .to_string(index=False))

if len(consensus) == 0:
    print('\n⚠ WARNING — no consensus hits. Consider:')
    print('  - Relaxing the Z-score cutoff to −2.0')
    print('  - Using a larger or more ion-channel-focused library')
    print('  - Checking that the docking box covers the correct pocket')
else:
    print(f'\n✓ PASS — {len(consensus)} consensus hits ready for AF3 validation')

In [ ]:
# Visualize top hit poses in 3D
hits = pd.read_csv(DOCKING_DIR.parent / 'top_hits.csv')
top5 = hits[hits['consensus_flag']].nsmallest(5, 'vina_zscore')

view = py3Dmol.view(width=900, height=500)
view.addModel(pdb_path.read_text(), 'pdb')
view.setStyle({'chain': 'A'}, {'cartoon': {'color': 'steelblue', 'opacity': 0.4}})
view.setStyle({'resn': LIGAND_RESNAME}, {'stick': {'colorscheme': 'orangeCarbon', 'radius': 0.3}})

colors = ['green', 'magenta', 'cyan', 'yellow', 'salmon']
for i, (_, row) in enumerate(top5.iterrows()):
    pose_file = DOCKING_DIR / f"{row['name']}_pose.pdbqt"
    if pose_file.exists():
        view.addModel(pose_file.read_text(), 'pdbqt')
        view.setStyle({'model': i+1}, {'stick': {'color': colors[i], 'radius': 0.25}})
        print(f"  {colors[i]:8s} = {row['name']}  (Z={row['vina_zscore']:.2f})")

view.zoomTo({'resn': LIGAND_RESNAME})
view.show()
print('\nOrange = A-967079 crystal reference')

---
## Section 8 — Phase 2: Generate AF3 co-fold jobs (top 20 hits)

For each consensus hit, generate:
1. An AF3 input JSON (TRPA1 TM domain 682–1119 + hit SMILES)
2. A SLURM job script

Upload these to the cluster and submit when on VPN.

In [ ]:
import json, textwrap
from Bio import SeqIO

# Load TRPA1 TM domain sequence (residues 682–1119, 0-indexed 681:1119)
fasta_path = REPO / 'protein' / 'sequences' / 'O75762_TRPA1.fasta'
record = next(SeqIO.parse(fasta_path, 'fasta'))
TM_SEQ = str(record.seq[681:1119])
print(f'TM domain: {len(TM_SEQ)} aa (residues 682–1119)')
print(TM_SEQ[:60] + '...')

hits = pd.read_csv(DOCKING_DIR.parent / 'top_hits.csv')
top20 = hits[hits['consensus_flag']].nsmallest(20, 'vina_zscore').reset_index(drop=True)
print(f'\nGenerating AF3 jobs for {len(top20)} compounds...')

json_dir = REPO / 'protein' / 'inputs' / 'trpa1_hits'
job_dir  = REPO / 'jobs' / 'trpa1_hits'
json_dir.mkdir(parents=True, exist_ok=True)
job_dir.mkdir(parents=True, exist_ok=True)

SLURM_TEMPLATE = textwrap.dedent("""\
    #!/bin/bash
    #SBATCH --job-name=AF3_{name}
    #SBATCH --output=jobs/logs/slurm-AF3_{name}_%j.out
    #SBATCH --error=jobs/logs/slurm-AF3_{name}_%j.err
    #SBATCH -p l20-gpu
    #SBATCH --account=faculty
    #SBATCH --gres=gpu:1
    #SBATCH --ntasks-per-node=1
    #SBATCH --cpus-per-task=10
    #SBATCH --mem=48GB

    mkdir -p /work/$USER/tmp /work/$USER/cache
    export APPTAINER_TMPDIR=/work/$USER/tmp
    export APPTAINER_CACHEDIR=/work/$USER/cache

    AF3_HOME=/opt/apps/community/alphafold3
    DATABASES_DIR=$AF3_HOME/AF3_databases
    MODEL_PARAMETERS_DIR=$AF3_HOME/AF3_models
    REPO_DIR=${{REPO_DIR:-/work/$USER/af3-pipeline}}
    JOB_NAME={name}

    mkdir -p /work/$USER/AF3_Input /work/$USER/AF3_Output/$JOB_NAME
    cp "$REPO_DIR/protein/inputs/trpa1_hits/{name}.json" /work/$USER/AF3_Input/{name}.json

    singularity exec --nv \\
        --bind /work/$USER/AF3_Input:/root/af_input \\
        --bind /work/$USER/AF3_Output/$JOB_NAME:/root/af_output \\
        --bind $MODEL_PARAMETERS_DIR:/root/models \\
        --bind $DATABASES_DIR:/root/public_databases \\
        --bind $AF3_HOME/AF3_Programs/alphafold3:/root/AF3 \\
        $AF3_HOME/AF3_v3.0.1.sif \\
        python /root/AF3/run_alphafold.py \\
        --flash_attention_implementation=xla \\
        --json_path=/root/af_input/{name}.json \\
        --model_dir=/root/models \\
        --db_dir=/root/public_databases \\
        --output_dir=/root/af_output

    echo done
    date
    """)

generated = []
for _, row in top20.iterrows():
    safe_name = f"trpa1_tm_{row['name'].replace(' ', '_').replace('/', '_')}"

    af3_input = {
        'name': safe_name,
        'modelSeeds': [1, 2, 3],
        'sequences': [
            {'protein': {'id': 'A', 'sequence': TM_SEQ}},
            {'ligand':  {'id': 'B', 'smiles': row['smiles']}}
        ],
        'dialect': 'alphafold3',
        'version': 2
    }

    json_file = json_dir / f'{safe_name}.json'
    json_file.write_text(json.dumps(af3_input, indent=2))

    job_file = job_dir / f'{safe_name}.job'
    job_file.write_text(SLURM_TEMPLATE.format(name=safe_name))

    generated.append(safe_name)

print(f'\nGenerated {len(generated)} AF3 input JSONs → {json_dir}')
print(f'Generated {len(generated)} SLURM job scripts → {job_dir}')
print('\nTo submit (on cluster with VPN):')
print('  cd /work/$USER/af3-pipeline')
for name in generated[:5]:
    print(f'  sbatch jobs/trpa1_hits/{name}.job')
if len(generated) > 5:
    print(f'  ... and {len(generated)-5} more')
print('\n✓ PASS — AF3 validation jobs ready')

---
## Section 9 — Phase 2: Analyze AF3 co-fold results

Run this after the AF3 jobs finish on the cluster and you've rsynced outputs back to `../output/`.

In [ ]:
import glob
from src.scoring.parse_af3 import load_summary_confidences

AF3_OUTPUT = REPO.parent / 'output'

records = []
for conf_file in sorted(AF3_OUTPUT.glob('trpa1_tm_*/*/summary_confidences.json')):
    job_name = conf_file.parts[-3]   # e.g. trpa1_tm_hit001
    seed_sample = conf_file.parts[-2]
    scores = load_summary_confidences(conf_file)
    records.append({
        'job': job_name,
        'seed_sample': seed_sample,
        'iptm': scores.get('iptm'),
        'ranking': scores.get('ranking_score'),
        'fraction_disordered': scores.get('fraction_disordered'),
        'has_clash': scores.get('has_clash'),
    })

if not records:
    print('No AF3 outputs found yet.')
    print(f'Expected pattern: {AF3_OUTPUT}/trpa1_tm_*/*/summary_confidences.json')
    print('rsync from cluster first, then rerun this cell.')
else:
    df = pd.DataFrame(records)

    # Best score per compound across all seeds
    best = df.groupby('job')['iptm'].max().reset_index().rename(columns={'iptm': 'iptm_best'})
    seed_agree = df.groupby('job').apply(lambda g: (g['iptm'] > 0.5).sum()).reset_index()
    seed_agree.columns = ['job', 'seeds_passing']
    summary = best.merge(seed_agree, on='job').sort_values('iptm_best', ascending=False)

    print(summary.to_string(index=False))

    validated = summary[(summary['iptm_best'] > 0.5) & (summary['seeds_passing'] >= 2)]
    print(f'\n✓ AF3-validated hits (iptm > 0.5 in ≥2 seeds): {len(validated)}')
    print(validated.to_string(index=False))

In [ ]:
# Visualize best AF3 model for top validated hit
if records and len(validated) > 0:
    top_job = validated.iloc[0]['job']
    best_model = sorted(AF3_OUTPUT.glob(f'{top_job}/*/model.cif'))[-1]
    print(f'Top validated hit: {top_job}')
    print(f'Model: {best_model}')

    view = py3Dmol.view(width=800, height=500)
    view.addModel(best_model.read_text(), 'cif')
    view.setStyle({'chain': 'A'}, {'cartoon': {'color': 'steelblue', 'opacity': 0.8}})
    view.setStyle({'chain': 'B'}, {'stick': {'colorscheme': 'greenCarbon', 'radius': 0.35}})
    view.zoomTo({'chain': 'B'})
    view.show()
    print('Blue = TRPA1 TM domain  |  Green = top hit compound (AF3 predicted pose)')
else:
    print('No validated hits yet — run AF3 jobs on cluster first.')

---
## Section 10 — Final shortlist

Merge Vina, GNINA, and AF3 scores into one table for the wet-lab handoff.

In [ ]:
top_hits = pd.read_csv(DOCKING_DIR.parent / 'top_hits.csv')

if records and len(validated) > 0:
    # Merge AF3 validation scores into hit table
    af3_scores = summary.copy()
    af3_scores['name'] = af3_scores['job'].str.replace('trpa1_tm_', '', regex=False)
    shortlist = top_hits.merge(af3_scores[['name', 'iptm_best', 'seeds_passing']], on='name', how='left')
    shortlist['af3_pass'] = (shortlist['iptm_best'] > 0.5) & (shortlist['seeds_passing'] >= 2)
    shortlist = shortlist[shortlist['af3_pass']].sort_values('vina_zscore')
else:
    # Show Phase 1 results only if AF3 not yet done
    shortlist = top_hits[top_hits['consensus_flag']].sort_values('vina_zscore')
    print('Note: AF3 validation scores not yet available — showing Phase 1 consensus hits only\n')

display_cols = ['name', 'smiles', 'vina_affinity', 'vina_zscore', 'gnina_score']
if 'iptm_best' in shortlist.columns:
    display_cols += ['iptm_best', 'seeds_passing']

print(f'=== TRPA1 SHORTLIST ({len(shortlist)} compounds) ===')
print(shortlist[display_cols].head(10).to_string(index=False))

shortlist.to_csv(DOCKING_DIR.parent / 'final_shortlist.csv', index=False)
print(f'\nSaved: results/trpa1/final_shortlist.csv')
print('\nNext step: order top 5–10 compounds for thermal shift + FLIPR assay')